# 02 Experiments: CNN Model Hyperparameter Search

This notebook runs multiple training configurations and tracks results with wandb.

In [1]:
import sys
from pathlib import Path

import wandb

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from src.data import build_labeled_dataset, make_sliding_windows, normalize_series
from src.train import train_model
from src.evaluate import evaluate_model

## Load Data

In [4]:
DATA_DIR = PROJECT_ROOT / "data"
LABELS_PATH = DATA_DIR / "combined_windows.json"

TRAIN_SERIES = [
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_53ea38.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_5f5533.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_825cc2.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_c6585a.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_fe7f93.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_disk_write_bytes_1ef3de.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_network_in_257a54.csv",
    DATA_DIR / "realAWSCloudwatch/grok_asg_anomaly.csv",
    DATA_DIR / "realAWSCloudwatch/iio_us-east-1_i-a2eb1cd9_NetworkIn.csv",
    DATA_DIR / "realAWSCloudwatch/rds_cpu_utilization_cc0c53.csv",
]

VAL_SERIES = [
    DATA_DIR / "realAWSCloudwatch/ec2_network_in_5abac7.csv",
    DATA_DIR / "realAWSCloudwatch/elb_request_count_8c0756.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_ac20cd.csv",
]

TEST_SERIES = [
    DATA_DIR / "realAWSCloudwatch/ec2_cpu_utilization_77c1ca.csv",
    DATA_DIR / "realAWSCloudwatch/ec2_disk_write_bytes_c0d644.csv",
    DATA_DIR / "realAWSCloudwatch/rds_cpu_utilization_e47b3b.csv",
]

WINDOW_SIZE = 24
HORIZON = 12

In [7]:
train_df = build_labeled_dataset(TRAIN_SERIES, LABELS_PATH)
train_df = normalize_series(train_df)
X_train, y_train = make_sliding_windows(train_df, WINDOW_SIZE, HORIZON)
print(f"Training set: {X_train.shape}, positive ratio: {y_train.mean():.4f}")

val_df = build_labeled_dataset(VAL_SERIES, LABELS_PATH)
val_df = normalize_series(val_df)
X_val, y_val = make_sliding_windows(val_df, WINDOW_SIZE, HORIZON)
print(f"Validation set: {X_val.shape}, positive ratio: {y_val.mean():.4f}")

test_df = build_labeled_dataset(TEST_SERIES, LABELS_PATH)
test_df = normalize_series(test_df)
X_test, y_test = make_sliding_windows(test_df, WINDOW_SIZE, HORIZON)
print(f"Test set: {X_test.shape}, positive ratio: {y_test.mean():.4f}")

Training set: (42465, 1, 24), positive ratio: 0.0949
Validation set: (12689, 1, 24), positive ratio: 0.1051
Test set: (11991, 1, 24), positive ratio: 0.1064


In [12]:
type(X_train[0][0][0])

numpy.float64

## Define Configurations

In [ ]:
BASE_CONFIG = {
    "model_name": "InceptionTime",
    "window_size": WINDOW_SIZE,
    "horizon": HORIZON,
    "batch_size": 32,
    "learning_rate": 0.001,
    "epochs": 50,
    "patience": 10,
    "pos_weight": 1.0,
}

CONFIGS = [
    {
        **BASE_CONFIG,
        "name": "baseline",
        "model_kwargs": {},
    },
    {
        **BASE_CONFIG,
        "name": "smaller_filters",
        "model_kwargs": {"nf": 16},
    },
    {
        **BASE_CONFIG,
        "name": "larger_filters",
        "model_kwargs": {"nf": 64},
    },
    {
        **BASE_CONFIG,
        "name": "deeper",
        "model_kwargs": {"depth": 9},
    },
    {
        **BASE_CONFIG,
        "name": "smaller_ks",
        "model_kwargs": {"ks": 21},
    },
    {
        **BASE_CONFIG,
        "name": "smaller_depth",
        "model_kwargs": {"depth": 3},
    },
]

## Run Experiments

In [ ]:
for config in CONFIGS:
    print(f"\n{'='*60}")
    print(f"Running: {config['name']}")
    print(f"{'='*60}\n")
    
    wandb.init(
        entity="tombik-warsaw-university-of-technology",
        project="Predictive-alerting-for-cloud-metrics",
        name=f"{config['model_name']}_{config['name']}",
        config=config
    )
    
    clf = train_model(X_train, y_train, X_val, y_val, config)
    metrics = evaluate_model(clf, X_val, y_val, prefix="val")

    wandb.finish()

## View Results

Results are tracked in wandb. Visit [wandb.ai](https://wandb.ai) to compare runs and select the best configuration.